# Cloud-9: Complexity Metrics for Assembly Index

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bordode/cloud9-assembly-index/blob/main/notebooks/Complexity_Metrics.ipynb)

**Objective**: Compute Approximate Entropy (ApEn), Sample Entropy (SampEn), and Detrended Fluctuation Analysis (DFA) for halo assembly classification.

**Based on**: Pincus (1991), Richman & Moorman (2000), Peng et al. (1994)

## 1. Setup

In [10]:
!pip install -q numpy scipy matplotlib seaborn scikit-learn torch
!pip install -q astropy h5py

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.stats import linregress
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 2. Approximate Entropy (ApEn)

Pincus (1991): Measure of regularity in time series.

**Algorithm:**
1. Form vectors x(i) = [u(i), u(i+1), ..., u(i+m-1)]
2. Compute C_i^m(r) = (number of x(j) within r of x(i)) / (N-m+1)
3. Phi^m(r) = average of ln(C_i^m(r))
4. ApEn(m,r,N) = Phi^m(r) - Phi^{m+1}(r)

In [9]:
def approximate_entropy(time_series, m=2, r=None):
    N = len(time_series)
    if r is None:
        r = 0.2 * np.std(time_series)

    def _phi(m):
        x = np.array([time_series[i:i+m] for i in range(N - m + 1)])
        C = np.zeros(len(x))
        for i in range(len(x)):
            dist = np.max(np.abs(x - x[i]), axis=1)
            C[i] = np.sum(dist <= r) / (N - m + 1)
        return np.mean(np.log(C + 1e-10))

    return _phi(m) - _phi(m + 1)

# Test
np.random.seed(42)
t = np.linspace(0, 10, 1000)
regular = np.sin(2 * np.pi * t) + 0.1 * np.random.randn(1000)
irregular = np.random.randn(1000)

apen_reg = approximate_entropy(regular)
apen_irr = approximate_entropy(irregular)
print(f'Regular: ApEn = {apen_reg:.4f}')
print(f'Irregular: ApEn = {apen_irr:.4f}')

Regular: ApEn = 0.6931
Irregular: ApEn = 1.6514


## 3. Sample Entropy (SampEn)

Richman & Moorman (2000): Bias-corrected version of ApEn.

In [5]:
def sample_entropy(time_series, m=2, r=None):
    N = len(time_series)
    if r is None:
        r = 0.2 * np.std(time_series)

    x_m = np.array([time_series[i:i+m] for i in range(N - m)])
    x_m1 = np.array([time_series[i:i+m+1] for i in range(N - m - 1)])

    B, A = 0, 0
    for i in range(len(x_m)):
        dist = np.max(np.abs(x_m - x_m[i]), axis=1)
        B += np.sum((dist <= r) & (np.arange(len(x_m)) != i))

    for i in range(len(x_m1)):
        dist = np.max(np.abs(x_m1 - x_m1[i]), axis=1)
        A += np.sum((dist <= r) & (np.arange(len(x_m1)) != i))

    if B == 0 or A == 0:
        return np.inf
    return -np.log(A / B)

sampen_reg = sample_entropy(regular)
sampen_irr = sample_entropy(irregular)
print(f'Regular: SampEn = {sampen_reg:.4f}')
print(f'Irregular: SampEn = {sampen_irr:.4f}')

Regular: SampEn = 0.6297
Irregular: SampEn = 2.1420


## 4. Detrended Fluctuation Analysis (DFA)

Peng et al. (1994): Quantify long-range correlations.

In [6]:
def dfa(time_series, min_window=4, max_window=None, order=1):
    N = len(time_series)
    if max_window is None:
        max_window = N // 4

    y = np.cumsum(time_series - np.mean(time_series))
    scales = np.unique(np.logspace(
        np.log10(min_window), np.log10(max_window), num=20
    ).astype(int))

    fluctuations = []
    for scale in scales:
        n_windows = N // scale
        rms_values = []
        for i in range(n_windows):
            window = y[i*scale:(i+1)*scale]
            x = np.arange(scale)
            coeffs = np.polyfit(x, window, order)
            trend = np.polyval(coeffs, x)
            rms = np.sqrt(np.mean((window - trend)**2))
            rms_values.append(rms)
        fluctuations.append(np.mean(rms_values))

    fluctuations = np.array(fluctuations)
    valid = (fluctuations > 0) & (scales > 0)
    slope, _, _, _, _ = linregress(
        np.log10(scales[valid]), np.log10(fluctuations[valid])
    )
    return slope, scales[valid], fluctuations[valid]

# Test
white = np.random.randn(1000)
pink = np.cumsum(np.random.randn(1000))
pink = pink / np.std(pink)

alpha_white, _, _ = dfa(white)
alpha_pink, _, _ = dfa(pink)
print(f'White noise: alpha = {alpha_white:.3f} (expected ~0.5)')
print(f'Pink noise: alpha = {alpha_pink:.3f} (expected ~1.0)')

White noise: alpha = 0.527 (expected ~0.5)
Pink noise: alpha = 1.518 (expected ~1.0)


## 5. Halo Assembly Analysis

Apply to simulated assembly histories.

In [8]:
def generate_assembly_history(atype='stochastic', n=1000):
    if atype == 'stochastic':
        base = np.linspace(1e11, 1e12, n)
        bursts = np.zeros(n)
        for _ in range(np.random.poisson(10)):
            bt = np.random.randint(0, n)
            bursts[bt:min(bt+20, n)] += np.random.exponential(0.2)
        return base * (1 + bursts)
    else:
        t = np.linspace(0, 10, n)
        sfh = np.exp(-(np.log(t+1) - np.log(6))**2 / 0.5)
        return 1e11 + np.cumsum(sfh) * 1e10 + np.random.randn(n) * 1e9

# Generate and analyze
stoch = [generate_assembly_history('stochastic') for _ in range(50)]
sec = [generate_assembly_history('secular') for _ in range(50)]

def analyze(histories):
    results = {'apen': [], 'sampen': [], 'dfa': []}
    for h in histories:
        norm = (h - np.mean(h)) / np.std(h)
        results['apen'].append(approximate_entropy(norm))
        results['sampen'].append(sample_entropy(norm))
        alpha, _, _ = dfa(norm)
        results['dfa'].append(alpha)
    return results

r_stoch = analyze(stoch)
r_sec = analyze(sec)

print('Stochastic:')
print(f"  ApEn: {np.mean(r_stoch['apen']):.3f} +/- {np.std(r_stoch['apen']):.3f}")
print(f"  SampEn: {np.mean(r_stoch['sampen']):.3f} +/- {np.std(r_stoch['sampen']):.3f}")
print(f"  DFA: {np.mean(r_stoch['dfa']):.3f} +/- {np.std(r_stoch['dfa']):.3f}")
print('\nSecular:')
print(f"  ApEn: {np.mean(r_sec['apen']):.3f} +/- {np.std(r_sec['apen']):.3f}")
print(f"  SampEn: {np.mean(r_sec['sampen']):.3f} +/- {np.std(r_sec['sampen']):.3f}")
print(f"  DFA: {np.mean(r_sec['dfa']):.3f} +/- {np.std(r_sec['dfa']):.3f}")

Stochastic:
  ApEn: 0.048 +/- 0.020
  SampEn: 0.023 +/- 0.008
  DFA: 1.718 +/- 0.081

Secular:
  ApEn: 0.001 +/- 0.000
  SampEn: 0.004 +/- 0.000
  DFA: 2.027 +/- 0.001


## 6. ML Classifier

Train neural network on complexity features.

In [11]:
# Prepare data
X_s = np.column_stack([r_stoch['apen'], r_stoch['sampen'], r_stoch['dfa']])
X_sec = np.column_stack([r_sec['apen'], r_sec['sampen'], r_sec['dfa']])
X = np.vstack([X_s, X_sec])
y = np.array([0]*50 + [1]*50)

X = np.nan_to_num(X, nan=0, posinf=10, neginf=-10)
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Model
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 16), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 2)
        )
    def forward(self, x):
        return self.net(x)

model = Classifier().to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

X_tr = torch.FloatTensor(X_train).to(device)
y_tr = torch.LongTensor(y_train).to(device)
X_te = torch.FloatTensor(X_test).to(device)
y_te = torch.LongTensor(y_test).to(device)

for epoch in range(100):
    model.train()
    opt.zero_grad()
    loss = criterion(model(X_tr), y_tr)
    loss.backward()
    opt.step()
    if (epoch+1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            acc = (model(X_te).argmax(1) == y_te).float().mean().item()
        print(f'Epoch {epoch+1}: Loss={loss.item():.4f}, Acc={acc:.2%}')

model.eval()
with torch.no_grad():
    final_acc = (model(X_te).argmax(1) == y_te).float().mean().item()
print(f'\nFinal Accuracy: {final_acc:.2%}')

Epoch 20: Loss=0.1304, Acc=93.33%
Epoch 40: Loss=0.0092, Acc=93.33%
Epoch 60: Loss=0.0023, Acc=96.67%
Epoch 80: Loss=0.0004, Acc=96.67%
Epoch 100: Loss=0.0012, Acc=96.67%

Final Accuracy: 96.67%


## Summary

- High SampEn: stochastic/merger-driven assembly (wake state)
- Low SampEn: secular/log-normal assembly (sleep state)
- A_c proxy: S_c = SampEn + DFA_alpha

Next: Apply to CAMELS data, validate against IllustrisTNG